In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)


In [ ]:
import random
random.seed(42)

# Bambaşka, çok daha zengin ve fantastik bir evren tasarlıyoruz
kisiler = ["astronot kamil", "siborg canan", "büyücü süleyman", "elf melisa", "dedektif rıza", "robot bobi", "kraliçe setenay", "göçebe tarkan"]
zamanlar = ["yeni çağda", "mavi dolunayda", "ışınlanma sırasında", "kıyamet sonrasında", "retro döneminde", "galaktik şafakta", "gece yarısında"]
yer_e = ["kara deliğe", "siber uzaya", "antik tapınağa", "mars üssüne", "büyülü ormana", "zaman tüneline", "uçan adaya"]
yer_de = ["kara delikte", "siber uzayda", "antik tapınakta", "mars üssünde", "büyülü ormanda", "zaman tünelinde", "uçan adada"]
isler = [("plazma topu", "fırlattı"), ("iksir", "kaynattı"), ("kod", "yazdı"), ("portal", "açtı"), ("kristal", "keşfetti"), ("roket", "fırlattı"), ("sinyal", "gönderdi")]
havalar = ["kozmik", "radyoaktif", "büyülü", "meteor fırtınalı", "yerçekimsiz", "aşırı sıcak", "gizemli"]
sifatlar = ["meraklı", "korkusuz", "panik halinde", "görünmez", "ışıl ışıl", "şaşkın", "melankolik"]

def cumle():
  k, z = random.choice(kisiler), random.choice(zamanlar)
  i = random.randrange(len(yer_e))
  nesne, fiil = random.choice(isler)
  kalip = random.randrange(6)
  if kalip == 0: return f"{z} {k} {yer_e[i]} gitti"
  if kalip == 1: return f"{z} {k} {yer_de[i]} {nesne} {fiil}"
  if kalip == 2: return f"{z} hava oldukça {random.choice(havalar)}"
  if kalip == 3: return f"{k} şu an çok {random.choice(sifatlar)}"
  if kalip == 4: return f"{z} {k} {yer_de[i]} çok {random.choice(sifatlar)} bir şekilde {nesne} {fiil}"
  return f"{z} {k} {yer_de[i]} {nesne} {fiil}"

# 7000 satırlık tamamen farklı bir evrene ait metin üretiyoruz
metin = "\n".join(cumle() for _ in range(7000))
print("Yeni Metin uzunluğu:", len(metin))
print(metin[:900])

Yeni Metin uzunluğu: 379125
yeni çağda siborg canan zaman tünelinde kod yazdı
mavi dolunayda elf melisa zaman tünelinde plazma topu fırlattı
retro döneminde siborg canan mars üssüne gitti
mavi dolunayda siborg canan siber uzayda çok meraklı bir şekilde kristal keşfetti
galaktik şafakta elf melisa zaman tünelinde çok görünmez bir şekilde roket fırlattı
kıyamet sonrasında elf melisa büyülü ormana gitti
galaktik şafakta hava oldukça radyoaktif
gece yarısında elf melisa antik tapınağa gitti
yeni çağda hava oldukça yerçekimsiz
dedektif rıza şu an çok ışıl ışıl
kıyamet sonrasında hava oldukça gizemli
retro döneminde robot bobi siber uzaya gitti
galaktik şafakta hava oldukça kozmik
gece yarısında hava oldukça meteor fırtınalı
mavi dolunayda robot bobi antik tapınakta kod yazdı
galaktik şafakta dedektif rıza zaman tüneline gitti
retro döneminde büyücü süleyman zaman tünelinde iksir kaynattı
kıyamet sonrasında göçebe tarkan


In [ ]:
harfler = sorted(set(metin))
SOZLUK = len(harfler)
harf2no = {u:i for i,u in enumerate(harfler)}
no2harf = {i:u for i,u in enumerate(harfler)}
def kodla(s):
  return[harf2no[h] for h in s ]

def coz(sayilar):
  return "".join(no2harf(i) for i in sayilar)

print("sözlük boyutu",SOZLUK)
print("harfler:","".join(harfler))

sözlük boyutu 30
harfler: 
 abcdefghiklmnoprstuvyzçöüğış


In [ ]:
veri= torch.tensor(kodla(metin))
n=int(0.9*len(veri))
egitim_veri=veri[:n]
dogrulama_veri=veri[n:]

In [ ]:
baglam=64
yıgın = 32

def yigin_al(bolum):
  d = egitim_veri if bolum == "egitim" else dogrulama_veri
  bas = torch.randint(len(d)-baglam -1 ,(yıgın,))
  x = torch.stack ([d[i:i+baglam] for i in bas])
  y = torch.stack([d[i+1:i+baglam+1] for i in bas])
  return x,y

x,y =yigin_al("egitim")

In [ ]:
from torch._prims_common import Dim
class MaskeliDikkat(nn.Module):
  def __init__(self,d_model,n_kafa):
    super().__init__()
    self.n_kafa = n_kafa
    self.d_k = d_model //n_kafa
    self.qkv = nn.Linear(d_model,3*d_model)
    self.cikis = nn.Linear(d_model,d_model)
    self.register_buffer("maske",torch.tril(torch.ones(baglam,baglam)))
  def forward(self,x):
    B,L,D =x.shape
    q,k,v = self.qkv(x).split(D,dim=2)
    q = q.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    k = k.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    v = v.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    skor = q@k.transpose(-2,-1)/math.sqrt(self.d_k)
    skor = skor.masked_fill(self.maske[:L,:L][None, None] == 0,float("-inf"))
    agirlik = F.softmax(skor,dim=-1)
    cikti = (agirlik @v).transpose(1,2).reshape(B,L,D)
    return self.cikis(cikti)


In [ ]:
class GPTBlok(nn.Module):
  def __init__ (self,d_model,n_kafa):
    super().__init__()
    self.norm1 = nn.LayerNorm(d_model)
    self.dikkat = MaskeliDikkat(d_model,n_kafa)
    self.norm2 = nn.LayerNorm(d_model)

    self.ffn = nn.Sequential(
        nn.Linear(d_model,4*d_model),
        nn.ReLU(),
        nn.Linear(4*d_model,d_model),
        nn.Dropout(0.5)
    )
  def forward(self ,x):
    x=x+self.dikkat(self.norm1(x))

    x=x+self.ffn(self.norm2(x))

    return x

class MiniGPT (nn.Module):
  def __init__(self,d_model=64,n_kafa=4,n_katman=2):
    super().__init__()
    self.harf_gomme =nn.Embedding(SOZLUK,d_model)
    self.konum_gomme =nn.Embedding(baglam,d_model)
    self.blok = nn.Sequential(*[GPTBlok(d_model,n_kafa) for _ in range(n_katman)])
    self.norm = nn.LayerNorm(d_model)
    self.cikis = nn.Linear(d_model,SOZLUK)

  def forward (self,x,hedef =None):
    B,L =x.shape
    h = self.harf_gomme(x)+self.konum_gomme(torch.arange(L))
    h= self.norm(self.blok(h))
    logits = self.cikis(h)
    if hedef is None:
      return logits
    loss = F.cross_entropy(logits.view(-1,SOZLUK),hedef.view(-1))
    return logits,loss
  @torch.no_grad()
  def uret(self, baslangic, n_harf, sicaklik=0.8):
      x = torch.tensor(kodla(baslangic)).unsqueeze(0)   # [L] -> [1, L]
      for _ in range(n_harf):
          x_kirp = x[:, -baglam:]
          logits = self(x_kirp)                          # hedef yokken sadece logits döner
          logits = logits[:, -1, :] / sicaklik
          olasilik = F.softmax(logits, dim=-1)
          sonraki = torch.multinomial(olasilik, num_samples=1)
          x = torch.cat((x, sonraki), dim=1)
      return coz(x[0].tolist())

In [ ]:
model =MiniGPT()


In [ ]:
import torch

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
egitim_kayip, dogrulama_kayip, adimlar = [], [], []

# Adım sayısını 1000'e çıkarıyoruz, böylece model karakter sıralamalarını çok daha net kavrayacak
for adim in range(1, 1001):
  model.train()
  x, y = yigin_al("egitim")

  _, kayip = model(x, y)
  optimizer.zero_grad()
  kayip.backward()
  optimizer.step()

  if adim % 100 == 0:
    model.eval()
    with torch.no_grad():
      _, d_kayip = model(*yigin_al("dogrulama"))
      adimlar.append(adim)
      egitim_kayip.append(kayip.item())
      dogrulama_kayip.append(d_kayip.item())
      print(f"Adım {adim:4d} | egitim kaybı {kayip.item():.3f} | dogrulama kaybı {d_kayip.item():.3f}")

Adım  100 | egitim kaybı 2.286 | dogrulama kaybı 2.260
Adım  200 | egitim kaybı 2.005 | dogrulama kaybı 1.969
Adım  300 | egitim kaybı 1.580 | dogrulama kaybı 1.489
Adım  400 | egitim kaybı 1.072 | dogrulama kaybı 0.949
Adım  500 | egitim kaybı 0.648 | dogrulama kaybı 0.564
Adım  600 | egitim kaybı 0.497 | dogrulama kaybı 0.427
Adım  700 | egitim kaybı 0.418 | dogrulama kaybı 0.351
Adım  800 | egitim kaybı 0.348 | dogrulama kaybı 0.299
Adım  900 | egitim kaybı 0.340 | dogrulama kaybı 0.261
Adım 1000 | egitim kaybı 0.298 | dogrulama kaybı 0.255


In [ ]:
def coz(sayilar):
  return "".join(no2harf[i] for i in sayilar)

model.eval()
for basla in ["bugün","ilgisu","tripcan"]:
  print(model.uret(basla,150))
  print("--"*80)

bugüne roket fırlattı
mavi dolunayda hava oldukça gizemli
büyücü süleyman şu an çok ran kamil antit  kapanik helinde
krastaliktalikt kapanik halinde
kraliç
----------------------------------------------------------------------------------------------------------------------------------------------------------------
ilgisuz
galaktik şafakta kraliçe setenay kara delikte kod yazdı
dedektif rıza büyülü ormanda kod yazdı
retro döneminde isinde çok korkusuz bir şekilde ir ke
----------------------------------------------------------------------------------------------------------------------------------------------------------------
tripcana antik tapınağa siborg canan antik tapınakta portal açtı
kıyamet sonrasında hava oldukça kozmik
ışınlanma sırasında göçebe tarkan büyülü ormanda hava
----------------------------------------------------------------------------------------------------------------------------------------------------------------
